## Managed Memory for Databricks Agents

### Installing Utilities and Libraries

In [ ]:
%pip install databricks-sdk=0.170.0 openai-agents==0.22.0 databricks-openai==0.17.1 databricks-agentbricks==0.6.0

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Generate the OAuth Token for Memory Creation Purposes

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_agentkit import AgentKitClient

workspace = WorkspaceClient()

headers = workspace.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = workspace.config.host.rstrip("/")

agentkit = AgentKitClient(workspace)

print("AgentKit client initialized!")

### Create the Managed Memory Store

In [ ]:
MEMORY_STORE_NAME = "carbonops-sustainability-memory"

# Create a new memory store
memory_store = agentkit.memory_stores.create(MEMORY_STORE_NAME)
print("New Lakebase-backed memory store created")

print("Memory Store:", MEMORY_STORE_NAME)

### Configure the Memory Scope

In [ ]:
# Partition for this demo user's memories
ACTOR_ID = "carbonops-manager-001"

print("Memory Store:", MEMORY_STORE_NAME)
print("Memory Scope:", ACTOR_ID)

### Store CarbonOps Sustainability Context

We'll store three independent memories: the manager's profile, the Ahmedabad facility's targets, and the manager's reporting preferences.

In [ ]:
memories = [
    {
        "path": "/profile/manager.md",
        "content": """
Alex Morgan is the Regional Sustainability Manager at CarbonOps.

Alex oversees manufacturing facilities in Ahmedabad,
Pune, and Chennai.

Alex focuses on reducing Scope 1 and Scope 2 emissions,
improving operational efficiency, and achieving
corporate sustainability targets.
""",
        "description": "CarbonOps sustainability manager profile"
    },
    {
        "path": "/facilities/ahmedabad.md",
        "content": """
The CarbonOps Ahmedabad manufacturing facility has
a monthly carbon emissions budget of 200 tCO2e.

Its carbon intensity target is 8 kg CO2e per unit.

The latest demonstration data shows:
- Total emissions: 180 tCO2e
- Production: 25,000 units

The facility prioritizes electricity efficiency,
diesel reduction, and renewable energy adoption.
""",
        "description": "Ahmedabad facility emissions and sustainability targets"
    },
    {
        "path": "/preferences/reporting.md",
        "content": """
Alex prefers concise sustainability reports.

Reports should include:
- Carbon emissions KPIs
- Carbon intensity
- Target comparisons
- Actionable recommendations

Use tables for comparisons and avoid unnecessary jargon.
""",
        "description": "Sustainability reporting preferences"
    }
]

for memory in memories:
    memory_store.put(
        actor_id=ACTOR_ID,
        path=memory["path"],
        content=memory["content"],
        description=memory["description"]
    )

    print("Saved:", memory["path"])

print("All CarbonOps memories saved!")

### Define the User Query

In [ ]:
user_question = (
    "How is our Ahmedabad factory performing "
    "against its sustainability targets?"
)

print("USER:", user_question)

### Retrieve the Relevant Managed Memories

In [ ]:
results = memory_store.search(
    actor_id=ACTOR_ID,
    query=user_question,
    limit=5
)

print("Memory search completed!")

In [ ]:
memory_context = "\n\n".join(
    result.memory.content
    for result in results
    if result.memory and result.memory.content
)

print("=== RETRIEVED MEMORY ===")
print(memory_context)

### Send Memory Context with the LLM API Call

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ChatMessage,
    ChatMessageRole
)

w = WorkspaceClient(
    host=workspace_host,
    token=token
)

user_prompt = f"""
You are the CarbonOps Sustainability Intelligence Assistant.

Use the following retrieved long-term memory to answer
the sustainability manager's question.

--- RETRIEVED MEMORY ---
{memory_context}
--- END MEMORY ---

Instructions:
- Use relevant information from memory.
- Calculate carbon intensity when sufficient data exists.
- Compare performance against the remembered targets.
- Provide concise and actionable recommendations.
- Treat remembered figures as historical or illustrative
  unless they are explicitly confirmed as current.
- Do not invent missing operational data.

User question:
{user_question}
"""

response = w.serving_endpoints.query(
    name="databricks-claude-sonnet-4-5",
    messages=[
        ChatMessage(
            role=ChatMessageRole.SYSTEM,
            content="You are a helpful sustainability assistant."
        ),
        ChatMessage(
            role=ChatMessageRole.USER,
            content=user_prompt
        )
    ],
    max_tokens=2048
)

print("=== CARBONOPS ASSISTANT RESPONSE ===")
print(response.choices[0].message.content)